## Modelo: Regresion Lineal

## Objetivo:
> El objetivo de este notebook es construir un modelo de recomendacion dque sugiera las categorias de productos mas relevantes para cada cliente basandose en el cluster al que fue asignado.

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [ ]:
category_map = joblib.load("../models/category_map.pkl")

model = joblib.load("../models/rfc_customer_classification.pkl")

df = pd.read_csv("../data/processed/olist_segmented_customers.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115878 entries, 0 to 115877
Data columns (total 12 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       115878 non-null  str    
 1   customer_id                    115878 non-null  str    
 2   order_purchase_timestamp       115878 non-null  str    
 3   customer_unique_id             115878 non-null  str    
 4   order_item_id                  115878 non-null  float64
 5   product_category_name_english  115878 non-null  str    
 6   payment_value                  115878 non-null  float64
 7   product_category_number        115878 non-null  int64  
 8   month                          115878 non-null  int64  
 9   day_of_week                    115878 non-null  int64  
 10  hour                           115878 non-null  int64  
 11  customer_type                  115878 non-null  int64  
dtypes: float64(2), int64(5), str(5)
memory us

In [11]:
print(category_map)

{0: 'housewares', 1: 'perfumery', 2: 'auto', 3: 'pet_shop', 4: 'stationery', 5: 'furniture_decor', 6: 'office_furniture', 7: 'garden_tools', 8: 'computers_accessories', 9: 'bed_bath_table', 10: 'toys', 11: 'construction_tools_construction', 12: 'telephony', 13: 'health_beauty', 14: 'electronics', 15: 'baby', 16: 'cool_stuff', 17: 'watches_gifts', 18: 'air_conditioning', 19: 'sports_leisure', 20: 'books_general_interest', 21: 'small_appliances', 22: 'food', 23: 'luggage_accessories', 24: 'fashion_underwear_beach', 25: 'christmas_supplies', 26: 'fashion_bags_accessories', 27: 'musical_instruments', 28: 'construction_tools_lights', 29: 'books_technical', 30: 'costruction_tools_garden', 31: 'home_appliances', 32: 'market_place', 33: 'agro_industry_and_commerce', 34: 'party_supplies', 35: 'home_confort', 36: 'cds_dvds_musicals', 37: 'industry_commerce_and_business', 38: 'consoles_games', 39: 'furniture_bedroom', 40: 'construction_tools_safety', 41: 'fixed_telephony', 42: 'drinks', 43: 'kitc

In [12]:
df_recommendation_map = df.groupby(['customer_type', 'product_category_number']).size().reset_index(name='count')
df_recommendation_map["product_category_name"] = df_recommendation_map["product_category_number"].map(category_map)
df_recommendation_map

,customer_type,product_category_number,count,product_category_name
0,0,0,2737,housewares
1,0,1,1172,perfumery
2,0,2,1562,auto
3,0,3,776,pet_shop
4,0,4,885,stationery
...,...,...,...,...
154,3,29,59,books_technical
155,3,30,24,costruction_tools_garden
156,3,31,39,home_appliances
157,3,32,12,market_place


In [13]:
recommendation_map = df_recommendation_map.sort_values(["customer_type", "count"], ascending=[True, False])
recommendation_map

,customer_type,product_category_number,count,product_category_name
9,0,9,4168,bed_bath_table
13,0,13,3652,health_beauty
5,0,5,3197,furniture_decor
19,0,19,3019,sports_leisure
8,0,8,2859,computers_accessories
...,...,...,...,...
156,3,31,39,home_appliances
149,3,24,36,fashion_underwear_beach
155,3,30,24,costruction_tools_garden
157,3,32,12,market_place


In [14]:
top_3_per_cluster = recommendation_map.groupby('customer_type').head(3)
top_3_per_cluster

,customer_type,product_category_number,count,product_category_name
9,0,9,4168,bed_bath_table
13,0,13,3652,health_beauty
5,0,5,3197,furniture_decor
59,1,38,1191,consoles_games
52,1,31,668,home_appliances
66,1,45,634,home_construction
101,2,9,3876,bed_bath_table
105,2,13,2945,health_beauty
111,2,19,2926,sports_leisure
134,3,9,3779,bed_bath_table


In [15]:
dict_recomendation_per_cluster = top_3_per_cluster.groupby("customer_type")["product_category_name"].apply(list).to_dict()
dict_recomendation_per_cluster

{0: ['bed_bath_table', 'health_beauty', 'furniture_decor'],
 1: ['consoles_games', 'home_appliances', 'home_construction'],
 2: ['bed_bath_table', 'health_beauty', 'sports_leisure'],
 3: ['bed_bath_table', 'health_beauty', 'computers_accessories']}

## Función de recomendación

In [24]:
def recommend(customer_data):

    cluster_id = model.predict(customer_data)[0]

    recommendation = dict_recomendation_per_cluster[cluster_id]

    return cluster_id, recommendation

print(recommend([[2, 1, 10, 50.0, 3]]))

(np.int64(0), ['bed_bath_table', 'health_beauty', 'furniture_decor'])


c:\Users\Usuario\miniconda3\envs\env_commerce_py313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
